# Reddit Sentiment Analyzer with Scavio API

Search Reddit for mentions of any brand, product, or topic, read top threads, and classify sentiment using the Scavio search API and LangChain. A free alternative to Brand24 and Mention.

**What you will learn:**
- Search Reddit posts with ScavioRedditSearch
- Read full threads and comments with ScavioRedditPost
- Classify sentiment (positive/negative/mixed) with an LLM agent
- Extract common complaints and praise

**Prerequisites:**
- Free Scavio API key (250 credits/month): https://dashboard.scavio.dev
- OpenAI API key

**Tools used:** ScavioRedditSearch, ScavioRedditPost

In [1]:
# pip install langchain langchain-openai langchain-scavio python-dotenv

In [2]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_scavio import ScavioRedditSearch, ScavioRedditPost

load_dotenv(override=True)

True

In [3]:
SYSTEM_PROMPT = """You are RedditSentimentAnalyzer, a brand sentiment research agent.

Workflow:
1. Take the user's brand, product, or topic name.
2. Call ScavioRedditSearch with 2-3 query variants:
   - "<brand> review"
   - "<brand> experience"
   - "<brand> worth it"
   Sort by "new" to get recent discussions.
3. Deduplicate results by post ID. Pick the top 3-5 most relevant threads.
4. Call ScavioRedditPost on each to read the full body and comments.
5. Analyze sentiment across all threads and produce:

   ## Reddit Sentiment Report: <brand/topic>

   ### Overall Sentiment: <Positive/Negative/Mixed>
   One-line summary of the general feeling.

   ### Praise (What People Love)
   Bulleted list of recurring positive themes with thread citations.

   ### Complaints (What People Hate)
   Bulleted list of recurring negative themes with thread citations.

   ### Notable Threads
   Top 3 threads with title, subreddit, URL, and one-line summary.

   ### Recommendation
   One paragraph: what this brand should do based on Reddit feedback.

Rules:
- Never invent thread titles, URLs, subreddits, or usernames.
- Call only ONE tool per step.
- Keep the final report under 400 words.
"""

In [4]:
def build_agent():
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    tools = [
        ScavioRedditSearch(max_results=10),
        ScavioRedditPost(),
    ]
    return create_agent(model, tools=tools, system_prompt=SYSTEM_PROMPT)

In [5]:
agent = build_agent()
result = agent.invoke({
    "messages": [{"role": "user", "content": "Notion productivity app"}]
})
print(result["messages"][-1].content)

## Reddit Sentiment Report: Notion Productivity App

### Overall Sentiment: Mixed
Reddit users appreciate Notion's powerful database and organizational capabilities but find its AI features limited and sometimes cumbersome for complex research workflows.

### Praise (What People Love)
- Powerful database queries and flexible organization for managing notes and projects. (r/research)
- Integration with AI for summaries on entries, helpful for basic note-taking and task management. (r/research)
- Popularity and familiarity, as many users already use Notion for productivity. (r/research)
- Effective as a central note system when combined with transcription tools like Otter.ai to improve information retention. (r/appdev)

### Complaints (What People Hate)
- AI feels bolted on rather than designed specifically for research or complex workflows. (r/research)
- Auto-linking between notes or papers is weak, requiring manual effort to connect ideas. (r/research)
- Not ideal for large-scale lite

## Next Steps

- Analyze any brand, product, or trending topic on Reddit
- Track sentiment over time by running monthly reports
- Compare sentiment across competitors
- Feed results into a brand monitoring dashboard

**Credits used:** ~4-6 per run (2-3 searches + 2-3 post reads)